In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

from scipy.optimize import least_squares
from scipy.ndimage import gaussian_filter1d
from scipy.integrate import simpson
from scipy.interpolate import PchipInterpolator
from scipy.special import voigt_profile

In [ ]:
# ============================================================
# SELECT XPS .XY FILE
# ============================================================


# Hide the root Tk window
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

input_file = filedialog.askopenfilename(
    title="Select an XPS .xy File",
    filetypes=[
        ("XY files", "*.xy"),
        ("All files", "*.*")
    ]
)

root.destroy()

if input_file == "":
    raise RuntimeError("No .xy file selected.")

input_file = Path(input_file)

print(f"Selected file:\n{input_file}")

In [ ]:
def specs_xy_to_csv(input_file, output_file=None):
    """
    Convert a SPECS SpecsLab .xy export into a clean two-column CSV.

    All metadata and comment lines beginning with '#' are removed.
    The first two numeric columns are retained:

        column 1 -> Binding Energy (eV)
        column 2 -> Intensity (counts/s)

    Additional numeric columns, such as transmission, are ignored.
    """

    input_path = Path(input_file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Could not find the input file: {input_path}"
        )

    if input_path.suffix.lower() != ".xy":
        raise ValueError(
            f"Expected an .xy file, but received: {input_path.suffix}"
        )

    if output_file is None:
        output_path = input_path.with_suffix(".csv")
    else:
        output_path = Path(output_file)

    spectrum_rows = []

    with input_path.open(
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as file:

        for line_number, line in enumerate(file, start=1):

            stripped = line.strip()

            # Ignore blank lines
            if not stripped:
                continue

            # Ignore all SPECS metadata/comment lines
            if stripped.startswith("#"):
                continue

            # Numeric data are whitespace separated
            columns = stripped.split()

            if len(columns) < 2:
                continue

            try:
                binding_energy = float(columns[0])
                intensity = float(columns[1])

            except ValueError:
                # Ignore any unexpected nonnumeric line
                continue

            if not (
                np.isfinite(binding_energy)
                and np.isfinite(intensity)
            ):
                continue

            spectrum_rows.append(
                (binding_energy, intensity)
            )

    if not spectrum_rows:
        raise ValueError(
            "No valid spectrum data were found. "
            "The file contained no non-comment rows with at least "
            "two numeric columns."
        )

    dataframe = pd.DataFrame(
        spectrum_rows,
        columns=[
            "Binding Energy (eV)",
            "Intensity",
        ],
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print("\nXY TO CSV CONVERSION COMPLETE\n")
    print(f"Input file       : {input_path}")
    print(f"Output file      : {output_path}")
    print(f"Points converted : {len(dataframe)}")
    print(
        "Binding-energy range: "
        f"{dataframe['Binding Energy (eV)'].min():.3f} to "
        f"{dataframe['Binding Energy (eV)'].max():.3f} eV"
    )

    print("\nFirst five converted rows:\n")
    print(dataframe.head())

    return dataframe, output_path

In [ ]:
df, csv_file = specs_xy_to_csv(input_file)

base_name = csv_file.stem

x_original = df["Binding Energy (eV)"].to_numpy(float)
y_original = df["Intensity"].to_numpy(float)

print(f"\nLoaded spectrum: {base_name}")
print(f"Number of data points: {len(x_original)}")

# GENERATE SHIRLEY BACKGROUND

In [ ]:
def auto_shirley_window(
    x,
    y,
    smooth_fraction=0.04,
    threshold_fraction=0.015,
    pad_fraction=0.06,
):
    """
    Propose a Shirley window around the detected peak envelope.

    The returned endpoints should be inspected manually before
    quantitative analysis.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    n = len(y_sorted)

    window = max(5, int(round(n * smooth_fraction)))

    if window % 2 == 0:
        window += 1

    largest_odd_window = n if n % 2 == 1 else n - 1
    window = min(window, largest_odd_window)

    half_window = window // 2
    kernel = np.ones(window, dtype=float) / window

    # Avoid zero-padding artifacts at the two data boundaries.
    y_padded = np.pad(
        y_sorted,
        pad_width=half_window,
        mode="edge"
    )

    y_smooth = np.convolve(
        y_padded,
        kernel,
        mode="valid"
    )

    edge_points = max(3, int(round(n * 0.08)))

    y_lowBE = np.median(y_smooth[:edge_points])
    y_highBE = np.median(y_smooth[-edge_points:])

    rough_baseline = np.linspace(
        y_lowBE,
        y_highBE,
        n
    )

    residual = y_smooth - rough_baseline
    positive_residual = np.maximum(residual, 0.0)

    residual_max = np.max(positive_residual)

    if residual_max <= 0:
        return float(x_sorted[0]), float(x_sorted[-1])

    active = np.flatnonzero(
        positive_residual
        > threshold_fraction * residual_max
    )

    if len(active) == 0:
        peak_index = int(np.argmax(positive_residual))
        active = np.array([peak_index])

    pad_points = max(2, int(round(n * pad_fraction)))

    left = max(0, active[0] - pad_points)
    right = min(n - 1, active[-1] + pad_points)

    return float(x_sorted[left]), float(x_sorted[right])

In [ ]:
def shirley_background(
    x,
    y,
    fit_min=None,
    fit_max=None,
    endpoint_points=5,
    max_iter=500,
    rtol=1e-8,
    atol=1e-6,
):
    """
    Calculate a classic iterative Shirley background over a limited
    near-peak XPS energy range.

    Parameters
    ----------
    x : array-like
        Binding energy values.
    y : array-like
        Intensity values, preferably corrected for analyzer transmission.
    fit_min, fit_max : float, optional
        Binding-energy limits of the Shirley region.
    endpoint_points : int
        Number of points used to estimate each endpoint intensity.
    max_iter : int
        Maximum number of Shirley iterations.
    rtol, atol : float
        Relative and absolute convergence tolerances.

    Returns
    -------
    background : ndarray
        Shirley background in the original data order. Values outside
        the selected range are NaN.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.ndim != 1 or y.ndim != 1:
        raise ValueError("x and y must be one-dimensional arrays.")

    if len(x) != len(y):
        raise ValueError("x and y must have the same length.")

    if len(x) < 3:
        raise ValueError("At least three data points are required.")

    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("x and y must contain only finite values.")

    # Sort from low binding energy to high binding energy.
    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    # Duplicate energy values make integration ambiguous.
    if np.any(np.diff(x_sorted) <= 0):
        raise ValueError("Binding-energy values must be unique.")

    if fit_min is None:
        fit_min = float(x_sorted[0])

    if fit_max is None:
        fit_max = float(x_sorted[-1])

    fit_min, fit_max = sorted((float(fit_min), float(fit_max)))

    fit_mask = (x_sorted >= fit_min) & (x_sorted <= fit_max)

    if np.count_nonzero(fit_mask) < 3:
        raise ValueError(
            "The Shirley fit range must contain at least three points."
        )

    x_fit = x_sorted[fit_mask]
    y_fit = y_sorted[fit_mask]

    n_endpoint = min(
        int(endpoint_points),
        max(1, len(y_fit) // 4)
    )

    # Median endpoint levels reduce sensitivity to individual noisy points.
    y_lowBE = float(np.median(y_fit[:n_endpoint]))
    y_highBE = float(np.median(y_fit[-n_endpoint:]))

    # Initial approximation.
    bg = np.full_like(y_fit, y_lowBE, dtype=float)

    converged = False

    for iteration in range(1, max_iter + 1):
        bg_old = bg.copy()

        # Do not clip negative values; retain the iterative Shirley equation.
        net = y_fit - bg_old

        # Cumulative trapezoidal integral from low BE to high BE.
        dx = np.diff(x_fit)
        trapezoids = 0.5 * (net[:-1] + net[1:]) * dx

        cumulative_area = np.concatenate(
            ([0.0], np.cumsum(trapezoids))
        )

        total_area = cumulative_area[-1]

        if (
            not np.isfinite(total_area)
            or abs(total_area) < np.finfo(float).eps
        ):
            raise RuntimeError(
                "The net Shirley area is zero or non-finite. "
                "Check the selected endpoints."
            )

        bg = (
            y_lowBE
            + (y_highBE - y_lowBE)
            * cumulative_area
            / total_area
        )

        if np.allclose(bg, bg_old, rtol=rtol, atol=atol):
            converged = True
            print(f"Converged after {iteration} iterations")
            break

    if not converged:
        raise RuntimeError(
            f"Shirley background did not converge after "
            f"{max_iter} iterations."
        )

    bg_sorted = np.full(y_sorted.shape, np.nan, dtype=float)
    bg_sorted[fit_mask] = bg

    background = np.empty_like(bg_sorted)
    background[order] = bg_sorted

    return background

In [ ]:
# Only divide by transmission if Count/s has not already been corrected.
y_for_analysis = y_original
# y_for_analysis = count_per_second / transmission

SHIRLEY_MIN_BE, SHIRLEY_MAX_BE = auto_shirley_window(
    x_original,
    y_for_analysis
)

print(
    f"Proposed Shirley window: "
    f"{SHIRLEY_MIN_BE:.2f} to {SHIRLEY_MAX_BE:.2f} eV"
)

shirley = shirley_background(
    x_original,
    y_for_analysis,
    fit_min=SHIRLEY_MIN_BE,
    fit_max=SHIRLEY_MAX_BE,
    endpoint_points=5
)

corrected = y_for_analysis - shirley

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    x_original,
    y_for_analysis,
    label="Spectrum"
)

plt.plot(
    x_original,
    shirley,
    label="Iterative Shirley background"
)

plt.plot(
    x_original,
    corrected,
    label="Background-corrected spectrum"
)

plt.axvline(
    SHIRLEY_MIN_BE,
    linestyle="--",
    label="Shirley endpoints"
)

plt.axvline(
    SHIRLEY_MAX_BE,
    linestyle="--"
)

y_top = np.max(y_original)

plt.text(SHIRLEY_MIN_BE, y_top * 0.95, f"{SHIRLEY_MIN_BE:.2f} eV",
         rotation=90, va="top", ha="right", fontsize=10, color="gray")

plt.text(SHIRLEY_MAX_BE, y_top * 0.95, f"{SHIRLEY_MAX_BE:.2f} eV",
         rotation=90, va="top", ha="left", fontsize=10, color="gray")
plt.title(f"{base_name} Background Correction")
plt.xlabel("Binding Energy (eV)")
plt.ylabel("Intensity")
plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()


# Advanced single-spectrum Nb 3d fitting pipeline

This notebook still fits **one CSV spectrum at a time** and retains the sequence:

1. adaptive Shirley background,
2. fully coupled nonlinear fit,
3. post-fit binding-energy shift,
4. area, uncertainty, residual, and stability reporting.

The revised model adds the following physically motivated improvements without
multiplying any fitted area, position, or width by an empirical correction ratio:

- all areas, positions, FWHMs, line-shape parameters, and residual-baseline
  parameters are optimized simultaneously;
- the oxide doublets use a true Voigt profile rather than a linear
  pseudo-Voigt approximation;
- the metallic asymmetric peak smoothing is defined in electron-volts rather
  than array-index units;
- a tightly bounded residual offset and slope absorb small Shirley-background
  mismatch without contributing to any species area;
- mild region balancing prevents the dominant Nb2O5 peak from completely
  controlling the objective;
- physically motivated soft position and shape priors are retained;
- multistart fitting uses a coarse screening stage followed by refinement;
- profile-likelihood and bootstrap diagnostics are available as optional cells;
- Shirley-background sensitivity remains explicitly reported.

No species curve is prevented from overlapping another species curve.


In [ ]:

# ============================================================
# PREPARE THE BACKGROUND-CORRECTED DATA
#
# The corrected data are not shifted upward. A bounded residual
# offset and slope are fitted simultaneously later. Those baseline
# terms are excluded from all reported species areas.
# ============================================================

x = np.asarray(x_original, dtype=float)
y = np.asarray(corrected, dtype=float)

mask = (
    (x >= SHIRLEY_MIN_BE)
    & (x <= SHIRLEY_MAX_BE)
    & np.isfinite(x)
    & np.isfinite(y)
)

x = x[mask]
y = y[mask]

order = np.argsort(x)
x = x[order]
y = y[order]

intensity_shift = 0.0
x_centered = x - np.mean(x)

print(f"Fitting window: {x.min():.3f} to {x.max():.3f} eV")
print("Artificial corrected-intensity shift: disabled")
print(f"Minimum corrected intensity: {np.min(y):.6f}")


In [ ]:

# ============================================================
# PHYSICAL CONSTANTS AND LINE SHAPES
# ============================================================

SPLIT = 2.72
RATIO = 2.0 / 3.0

# Retained only to preserve the existing metallic-width convention.
METAL_FWHM_CALIBRATION = np.mean([
    0.81600 / 1.72100,
    0.63663 / 1.34900,
    0.60000 / 1.28700,
    0.58000 / 1.25200,
    0.59000 / 1.25600,
    0.55000 / 1.12500,
    0.57000 / 1.22500,
])

def metal_internal_to_fwhm(internal_width):
    return float(internal_width) * METAL_FWHM_CALIBRATION

def metal_fwhm_to_internal(fwhm):
    return float(fwhm) / METAL_FWHM_CALIBRATION


def true_voigt_from_fwhm_eta(x_values, center, fwhm, eta):
    """
    True Voigt profile parameterized by an approximate total FWHM and
    Lorentzian fraction eta.

    The Olivero-Longbothum Voigt-width approximation is inverted so that
    the resulting Voigt profile has approximately the requested total FWHM.
    """
    x_values = np.asarray(x_values, dtype=float)
    fwhm = max(float(fwhm), 1e-8)
    eta = float(np.clip(eta, 0.0, 1.0))

    lorentz_fwhm = max(eta * fwhm, 1e-8)

    inside = (
        (fwhm - 0.5346 * lorentz_fwhm) ** 2
        - 0.2166 * lorentz_fwhm ** 2
    )
    gaussian_fwhm = np.sqrt(max(inside, 1e-12))

    sigma = gaussian_fwhm / 2.354820045
    gamma = lorentz_fwhm / 2.0

    profile = voigt_profile(x_values - center, sigma, gamma)
    maximum = np.max(profile)

    if not np.isfinite(maximum) or maximum <= 0:
        return np.zeros_like(x_values)

    return profile / maximum


def oxide_doublet_height(x_values, height, center_5_2, shared_fwhm, eta):
    p_5_2 = height * true_voigt_from_fwhm_eta(
        x_values,
        center_5_2,
        shared_fwhm,
        eta,
    )

    p_3_2 = height * RATIO * true_voigt_from_fwhm_eta(
        x_values,
        center_5_2 + SPLIT,
        shared_fwhm,
        eta,
    )

    return p_5_2 + p_3_2


def LA_peak(
    x_values,
    height,
    center,
    internal_width,
    alpha=1.2,
    beta=5.0,
    smoothing_fraction=0.12,
):
    """
    Asymmetric Lorentzian-like metallic line shape.

    Smoothing is converted from electron-volts to array points, so the
    physical shape does not change when the input sampling density changes.
    """
    x_values = np.asarray(x_values, dtype=float)

    gamma = max(float(internal_width) / 2.0, 1e-10)
    lorentz = 1.0 / (1.0 + ((x_values - center) / gamma) ** 2)

    profile = np.zeros_like(x_values, dtype=float)
    low_be = x_values <= center
    profile[low_be] = lorentz[low_be] ** beta
    profile[~low_be] = lorentz[~low_be] ** alpha

    if len(x_values) > 1 and smoothing_fraction > 0:
        sorted_unique = np.sort(np.unique(x_values))
        if len(sorted_unique) > 1:
            step = float(np.median(np.diff(sorted_unique)))
            smoothing_sigma_ev = max(
                smoothing_fraction * metal_internal_to_fwhm(internal_width),
                0.01,
            )
            sigma_points = smoothing_sigma_ev / max(abs(step), 1e-12)
            profile = gaussian_filter1d(
                profile,
                sigma=max(sigma_points, 1e-6),
                mode="nearest",
            )

    maximum = np.max(profile)
    if not np.isfinite(maximum) or maximum <= 0:
        return np.zeros_like(x_values)

    return height * profile / maximum


def metal_doublet_height(x_values, height, center_5_2, internal_width):
    p_5_2 = LA_peak(
        x_values,
        height,
        center_5_2,
        internal_width,
    )

    p_3_2 = LA_peak(
        x_values,
        height * RATIO,
        center_5_2 + SPLIT,
        internal_width,
    )

    return p_5_2 + p_3_2


print(f"Metal FWHM calibration factor = {METAL_FWHM_CALIBRATION:.6f}")
print("Oxide line shape: true Voigt")
print("Metal smoothing: defined in eV, not array-index units")


In [ ]:

# ============================================================
# AREA-NORMALIZED COMPONENTS
#
# Every fitted area is defined as the numerical integral of that
# species curve over the selected fitting window.
# ============================================================

AREA_GRID = np.linspace(np.min(x), np.max(x), 12000)

def normalize_on_area_grid(unit_curve_on_area_grid):
    area = float(simpson(unit_curve_on_area_grid, x=AREA_GRID))

    if not np.isfinite(area) or abs(area) < 1e-15:
        raise ValueError("Component normalization area is zero or non-finite.")

    return abs(area)


def metal_component(x_values, area, center_5_2, internal_width):
    normalization_curve = metal_doublet_height(
        AREA_GRID,
        1.0,
        center_5_2,
        internal_width,
    )
    normalization = normalize_on_area_grid(normalization_curve)

    requested_curve = metal_doublet_height(
        np.asarray(x_values, dtype=float),
        1.0,
        center_5_2,
        internal_width,
    )

    return float(area) * requested_curve / normalization


def oxide_component(
    x_values,
    area,
    center_5_2,
    shared_oxide_fwhm,
    eta,
):
    normalization_curve = oxide_doublet_height(
        AREA_GRID,
        1.0,
        center_5_2,
        shared_oxide_fwhm,
        eta,
    )
    normalization = normalize_on_area_grid(normalization_curve)

    requested_curve = oxide_doublet_height(
        np.asarray(x_values, dtype=float),
        1.0,
        center_5_2,
        shared_oxide_fwhm,
        eta,
    )

    return float(area) * requested_curve / normalization


In [ ]:

# ============================================================
# FULLY COUPLED SIMULTANEOUS MODEL
#
# Parameters:
#  0  Nb metal area
#  1  Nb metal 3d5/2 position
#  2  Nb metal internal width
#  3  shared oxide FWHM
#  4  shared oxide Voigt eta
#  5  NbO area
#  6  NbO 3d5/2 position
#  7  NbO2 area
#  8  NbO2 3d5/2 position
#  9  Nb2O5 area
# 10  Nb2O5 3d5/2 position
# 11  residual baseline offset
# 12  residual baseline slope
#
# Baseline parameters are fitted simultaneously but are not included
# in any species area.
# ============================================================

def component_curves(x_values, p):
    shared_oxide_fwhm = p[3]
    shared_oxide_eta = p[4]

    metal = metal_component(
        x_values,
        area=p[0],
        center_5_2=p[1],
        internal_width=p[2],
    )

    nbo = oxide_component(
        x_values,
        area=p[5],
        center_5_2=p[6],
        shared_oxide_fwhm=shared_oxide_fwhm,
        eta=shared_oxide_eta,
    )

    nbo2 = oxide_component(
        x_values,
        area=p[7],
        center_5_2=p[8],
        shared_oxide_fwhm=shared_oxide_fwhm,
        eta=shared_oxide_eta,
    )

    nb2o5 = oxide_component(
        x_values,
        area=p[9],
        center_5_2=p[10],
        shared_oxide_fwhm=shared_oxide_fwhm,
        eta=shared_oxide_eta,
    )

    return metal, nbo, nbo2, nb2o5


def residual_baseline(x_values, p):
    x_values = np.asarray(x_values, dtype=float)
    return p[11] + p[12] * (x_values - np.mean(x))


def total_model(x_values, p):
    components = np.sum(component_curves(x_values, p), axis=0)
    return components + residual_baseline(x_values, p)


In [ ]:

# ============================================================
# STARTING VALUES, SOFT PRIORS, AND SAFETY BOUNDS
# ============================================================

def pchip_integral(x_values, y_values, x_min=None, x_max=None):
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)

    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[valid]
    y_values = y_values[valid]

    order = np.argsort(x_values)
    x_values = x_values[order]
    y_values = y_values[order]

    x_values, unique_idx = np.unique(x_values, return_index=True)
    y_values = y_values[unique_idx]

    if x_min is None:
        x_min = x_values[0]
    if x_max is None:
        x_max = x_values[-1]

    x_min, x_max = sorted((float(x_min), float(x_max)))
    spline = PchipInterpolator(x_values, y_values, extrapolate=False)
    anti = spline.antiderivative()

    return float(abs(anti(x_max) - anti(x_min)))


total_area_guess = max(pchip_integral(x, y), 1.0)

CENTER_PRIORS = {
    "metal": {"center": 202.20, "sigma": 0.30},
    "nbo":   {"center": 203.70, "sigma": 1.00},
    "nbo2":  {"center": 206.20, "sigma": 1.00},
    "nb2o5": {"center": 207.40, "sigma": 0.40},
}

REFERENCE_METAL_BE = CENTER_PRIORS["metal"]["center"]

COMMON_SHIFT_LIMIT = 2.0
RELATIVE_BOUND_MULTIPLIER = 2.5

def center_bounds(species):
    target = CENTER_PRIORS[species]["center"]
    sigma_prior = CENTER_PRIORS[species]["sigma"]
    half_width = COMMON_SHIFT_LIMIT + RELATIVE_BOUND_MULTIPLIER * sigma_prior
    return target - half_width, target + half_width


metal_fwhm_guess = 0.75
metal_internal_guess = metal_fwhm_to_internal(metal_fwhm_guess)

shared_oxide_fwhm_guess = 1.12
shared_oxide_eta_guess = 0.25

USE_SHAPE_PRIORS = True
SHAPE_PRIORS = {
    "metal_fwhm": {"center": 0.75, "sigma": 0.25},
    "oxide_fwhm": {"center": 1.12, "sigma": 0.25},
    "oxide_eta":  {"center": 0.25, "sigma": 0.30},
}

PRIOR_STRENGTH = 1.0

initial_fractions = np.array([
    0.14454,
    0.02357,
    0.03829,
    0.79360,
], dtype=float)
initial_fractions /= initial_fractions.sum()
initial_areas = total_area_guess * initial_fractions

# Automatic limits for small residual-baseline mismatch.
edge_count_for_bounds = max(6, int(round(0.12 * len(y))))
edge_for_bounds = np.concatenate([
    y[:edge_count_for_bounds],
    y[-edge_count_for_bounds:],
])

edge_scale = max(
    1e-6,
    1.4826 * np.median(
        np.abs(edge_for_bounds - np.median(edge_for_bounds))
    ),
    0.01 * max(float(np.ptp(y)), 1.0),
)

baseline_offset_limit = 3.0 * edge_scale
baseline_slope_limit = baseline_offset_limit / max(float(np.ptp(x)), 1e-6)

labels = ["Nb metal", "NbO", "NbO2", "Nb2O5"]

print("\nINITIAL AREA GUESSES\n")
for label, fraction, area in zip(labels, initial_fractions, initial_areas):
    print(f"{label:<8} fraction = {fraction:.5f}, area = {area:.2f}")

p0 = np.array([
    initial_areas[0],
    CENTER_PRIORS["metal"]["center"],
    metal_internal_guess,

    shared_oxide_fwhm_guess,
    shared_oxide_eta_guess,

    initial_areas[1],
    CENTER_PRIORS["nbo"]["center"],

    initial_areas[2],
    CENTER_PRIORS["nbo2"]["center"],

    initial_areas[3],
    CENTER_PRIORS["nb2o5"]["center"],

    0.0,
    0.0,
], dtype=float)

metal_lo, metal_hi = center_bounds("metal")
nbo_lo, nbo_hi = center_bounds("nbo")
nbo2_lo, nbo2_hi = center_bounds("nbo2")
nb2o5_lo, nb2o5_hi = center_bounds("nb2o5")

lower = np.array([
    0.0,
    metal_lo,
    metal_fwhm_to_internal(0.35),

    0.70,
    0.0,

    0.0,
    nbo_lo,

    0.0,
    nbo2_lo,

    0.0,
    nb2o5_lo,

    -baseline_offset_limit,
    -baseline_slope_limit,
], dtype=float)

upper = np.array([
    2.0 * total_area_guess,
    metal_hi,
    metal_fwhm_to_internal(1.20),

    1.55,
    1.0,

    2.0 * total_area_guess,
    nbo_hi,

    2.0 * total_area_guess,
    nbo2_hi,

    2.0 * total_area_guess,
    nb2o5_hi,

    baseline_offset_limit,
    baseline_slope_limit,
], dtype=float)

print(f"\nInitial total area: {total_area_guess:.2f}")
print(f"Residual offset limit: ±{baseline_offset_limit:.6g}")
print(f"Residual slope limit: ±{baseline_slope_limit:.6g} intensity/eV")
print("\nFinal shifted-position soft priors:")
for species, config in CENTER_PRIORS.items():
    print(
        f"{species:<7}: {config['center']:.3f} ± "
        f"{config['sigma']:.3f} eV"
    )


In [ ]:

# ============================================================
# MULTI-START, FULLY COUPLED SIMULTANEOUS FIT
# ============================================================

edge_count = max(6, int(round(0.12 * len(y))))
edge_values = np.concatenate([y[:edge_count], y[-edge_count:]])
edge_differences = np.diff(edge_values)

if len(edge_differences) > 0:
    diff_median = np.median(edge_differences)
    noise_sigma = (
        1.4826
        * np.median(np.abs(edge_differences - diff_median))
        / np.sqrt(2.0)
    )
else:
    noise_sigma = np.nan

fallback_noise = max(
    1e-6,
    0.01 * max(float(np.ptp(y)), 1.0),
)

if not np.isfinite(noise_sigma) or noise_sigma <= 0:
    noise_sigma = fallback_noise

sigma = np.full_like(y, noise_sigma, dtype=float)

AREA_INDICES = (0, 5, 7, 9)
CENTER_INDICES = (1, 6, 8, 10)

# Mildly balance chemically informative regions so the dominant Nb2O5
# maximum does not completely determine the objective.
USE_REGION_BALANCING = True
REGION_WEIGHT_CAP = (0.70, 2.00)

def make_region_weights(x_values):
    if not USE_REGION_BALANCING:
        return np.ones_like(x_values, dtype=float)

    x_values = np.asarray(x_values, dtype=float)

    # Broad regions; overlap is still fully allowed in the model.
    boundaries = np.array([
        203.0,
        205.0,
        206.9,
        208.8,
    ])

    region_id = np.digitize(x_values, boundaries)
    weights = np.ones_like(x_values, dtype=float)

    unique_regions = np.unique(region_id)
    target_count = len(x_values) / max(len(unique_regions), 1)

    for rid in unique_regions:
        region_mask = region_id == rid
        count = max(np.count_nonzero(region_mask), 1)
        weight = np.sqrt(target_count / count)
        weights[region_mask] = weight

    weights = np.clip(
        weights,
        REGION_WEIGHT_CAP[0],
        REGION_WEIGHT_CAP[1],
    )

    # Keep the mean squared weight near one.
    weights /= np.sqrt(np.mean(weights ** 2))
    return weights


region_weights = make_region_weights(x)


def shifted_positions_from_parameters(p):
    shift = REFERENCE_METAL_BE - p[1]
    return np.array([p[i] + shift for i in CENTER_INDICES], dtype=float)


def position_prior_residuals(p):
    shifted = shifted_positions_from_parameters(p)

    return np.array([
        (
            shifted[1] - CENTER_PRIORS["nbo"]["center"]
        ) / CENTER_PRIORS["nbo"]["sigma"],
        (
            shifted[2] - CENTER_PRIORS["nbo2"]["center"]
        ) / CENTER_PRIORS["nbo2"]["sigma"],
        (
            shifted[3] - CENTER_PRIORS["nb2o5"]["center"]
        ) / CENTER_PRIORS["nb2o5"]["sigma"],
    ], dtype=float)


def shape_prior_residuals(p):
    if not USE_SHAPE_PRIORS:
        return np.empty(0, dtype=float)

    metal_fwhm = metal_internal_to_fwhm(p[2])

    return np.array([
        (
            metal_fwhm - SHAPE_PRIORS["metal_fwhm"]["center"]
        ) / SHAPE_PRIORS["metal_fwhm"]["sigma"],
        (
            p[3] - SHAPE_PRIORS["oxide_fwhm"]["center"]
        ) / SHAPE_PRIORS["oxide_fwhm"]["sigma"],
        (
            p[4] - SHAPE_PRIORS["oxide_eta"]["center"]
        ) / SHAPE_PRIORS["oxide_eta"]["sigma"],
    ], dtype=float)


def baseline_prior_residuals(p):
    return np.array([
        p[11] / max(baseline_offset_limit / 2.0, 1e-12),
        p[12] / max(baseline_slope_limit / 2.0, 1e-12),
    ], dtype=float)


def data_residual_vector(p):
    raw = (total_model(x, p) - y) / sigma
    return region_weights * raw


def residual_vector(p):
    return np.concatenate([
        data_residual_vector(p),
        PRIOR_STRENGTH * position_prior_residuals(p),
        PRIOR_STRENGTH * shape_prior_residuals(p),
        0.50 * baseline_prior_residuals(p),
    ])


rng = np.random.default_rng(42)
starts = [p0.copy()]

dirichlet_alpha = np.array([3.0, 1.0, 1.0, 8.0])
NUMBER_RANDOM_STARTS = 29

for _ in range(NUMBER_RANDOM_STARTS):
    trial = p0.copy()

    random_fractions = rng.dirichlet(dirichlet_alpha)
    random_total_area = total_area_guess * rng.lognormal(
        mean=0.0,
        sigma=0.15,
    )
    random_areas = random_total_area * random_fractions

    for idx, area in zip(AREA_INDICES, random_areas):
        trial[idx] = area

    common_offset = np.clip(
        rng.normal(0.0, 0.45),
        -COMMON_SHIFT_LIMIT,
        COMMON_SHIFT_LIMIT,
    )

    trial[1] = CENTER_PRIORS["metal"]["center"] + common_offset
    trial[6] = (
        CENTER_PRIORS["nbo"]["center"]
        + common_offset
        + rng.normal(0.0, 0.45 * CENTER_PRIORS["nbo"]["sigma"])
    )
    trial[8] = (
        CENTER_PRIORS["nbo2"]["center"]
        + common_offset
        + rng.normal(0.0, 0.45 * CENTER_PRIORS["nbo2"]["sigma"])
    )
    trial[10] = (
        CENTER_PRIORS["nb2o5"]["center"]
        + common_offset
        + rng.normal(0.0, 0.45 * CENTER_PRIORS["nb2o5"]["sigma"])
    )

    trial[2] = metal_fwhm_to_internal(
        np.clip(
            rng.normal(metal_fwhm_guess, 0.15),
            0.35,
            1.20,
        )
    )

    trial[3] = np.clip(
        rng.normal(shared_oxide_fwhm_guess, 0.16),
        lower[3],
        upper[3],
    )

    trial[4] = np.clip(
        rng.normal(shared_oxide_eta_guess, 0.18),
        lower[4],
        upper[4],
    )

    trial[11] = rng.normal(0.0, baseline_offset_limit / 3.0)
    trial[12] = rng.normal(0.0, baseline_slope_limit / 3.0)

    trial = np.clip(trial, lower + 1e-10, upper - 1e-10)
    starts.append(trial)


# -------------------------------
# Stage 1: coarse screening
# -------------------------------

coarse_candidates = []

print(f"Beginning coarse fitting of {len(starts)} starts...")

for start_number, start in enumerate(starts):
    try:
        result = least_squares(
            residual_vector,
            start,
            bounds=(lower, upper),
            method="trf",
            x_scale="jac",
            loss="soft_l1",
            f_scale=1.0,
            max_nfev=3000,
            ftol=1e-7,
            xtol=1e-7,
            gtol=1e-7,
        )

        if np.all(np.isfinite(result.x)):
            full_score = float(np.mean(residual_vector(result.x) ** 2))
            data_score = float(np.mean(data_residual_vector(result.x) ** 2))

            coarse_candidates.append(
                (full_score, data_score, start_number, result)
            )

    except (ValueError, RuntimeError, FloatingPointError):
        continue

    if (start_number + 1) % 5 == 0:
        print(f"Completed {start_number + 1} of {len(starts)} starts")


if not coarse_candidates:
    raise RuntimeError("All coarse simultaneous fitting starts failed.")

coarse_candidates.sort(key=lambda item: item[0])


# -------------------------------
# Stage 2: refine best solutions
# -------------------------------

NUMBER_TO_REFINE = min(5, len(coarse_candidates))
refined_candidates = []

print(f"\nRefining the best {NUMBER_TO_REFINE} solutions...")

for rank, coarse_candidate in enumerate(
    coarse_candidates[:NUMBER_TO_REFINE],
    start=1,
):
    _, _, original_start_number, coarse_result = coarse_candidate

    try:
        refined_result = least_squares(
            residual_vector,
            coarse_result.x,
            bounds=(lower, upper),
            method="trf",
            x_scale="jac",
            loss="soft_l1",
            f_scale=1.0,
            max_nfev=15000,
            ftol=1e-9,
            xtol=1e-9,
            gtol=1e-9,
        )

        if np.all(np.isfinite(refined_result.x)):
            refined_full_score = float(
                np.mean(residual_vector(refined_result.x) ** 2)
            )
            refined_data_score = float(
                np.mean(data_residual_vector(refined_result.x) ** 2)
            )

            refined_candidates.append(
                (
                    refined_full_score,
                    refined_data_score,
                    original_start_number,
                    refined_result,
                )
            )

            print(
                f"Refinement {rank}: full score = "
                f"{refined_full_score:.8g}"
            )

    except (ValueError, RuntimeError, FloatingPointError):
        continue


candidates = refined_candidates if refined_candidates else coarse_candidates
candidates.sort(key=lambda item: item[0])

best_score, best_data_score, best_start_number, best_result = candidates[0]
popt = best_result.x

print("\nFINAL MULTI-START RESULTS\n")
print(f"Estimated point-noise sigma: {noise_sigma:.6g}")
print(f"Region balancing enabled: {USE_REGION_BALANCING}")
print(f"Successful coarse starts: {len(coarse_candidates)} / {len(starts)}")
print(f"Successful refined fits: {len(refined_candidates)} / {NUMBER_TO_REFINE}")
print(f"Best original starting trial: {best_start_number}")
print(f"Best full penalized score: {best_score:.8g}")
print(f"Best data-only reduced score: {best_data_score:.8g}")
print(f"Residual baseline offset: {popt[11]:+.6g}")
print(f"Residual baseline slope: {popt[12]:+.6g} intensity/eV")

print("\nBest available solutions:")
for rank, (full_score, data_score, start_number, result) in enumerate(
    candidates[:5],
    start=1,
):
    print(
        f"{rank}: full = {full_score:.8g}, "
        f"data = {data_score:.8g}, "
        f"start = {start_number}, "
        f"nfev = {result.nfev}"
    )


In [ ]:

# ============================================================
# UNSHIFTED SIMULTANEOUS RESULTS
# ============================================================

component_names = ["Nb metal", "NbO", "NbO2", "Nb2O5"]
area_indices = [0, 5, 7, 9]
center_indices = [1, 6, 8, 10]

fitted_areas = np.array([popt[i] for i in area_indices])
unshifted_centers = np.array([popt[i] for i in center_indices])

metal_fwhm_reported = metal_internal_to_fwhm(popt[2])
shared_oxide_fwhm = popt[3]
shared_oxide_eta = popt[4]

print(f"\n{base_name} UNSHIFTED SIMULTANEOUS RESULTS\n")

for name, area, center in zip(
    component_names,
    fitted_areas,
    unshifted_centers,
):
    reported_fwhm = (
        metal_fwhm_reported
        if name == "Nb metal"
        else shared_oxide_fwhm
    )
    print(
        f"{name:<8}: Area = {area:.2f}, "
        f"3d5/2 = {center:.3f} eV, "
        f"FWHM = {reported_fwhm:.3f} eV"
    )

print(f"\nTotal fitted species area = {fitted_areas.sum():.2f}")
print(f"Residual baseline offset = {popt[11]:+.6g}")
print(f"Residual baseline slope = {popt[12]:+.6g} intensity/eV")
print(f"Shared oxide Voigt eta = {shared_oxide_eta:.4f}")
print("Doublet separation = 2.72 eV")
print("3d5/2 : 3d3/2 area ratio = 3 : 2")


# Binding-energy shift

The simultaneous fit is completed first. Then the same shift used in the original
notebook is applied:

```python
shift = 202.200 - fitted_metal_position
```

The same constant is added to every species position and to the plotted energy axis.

In [ ]:
# ============================================================
# APPLY THE ORIGINAL ENERGY-SHIFT PROCESS AFTER FITTING
# ============================================================

energy_shift = REFERENCE_METAL_BE - popt[1]

shifted_centers = unshifted_centers + energy_shift
x_shifted = x + energy_shift

metal_shifted, nbo_shifted, nbo2_shifted, nb2o5_shifted = shifted_centers

print(f"\n{base_name} SHIFTED BINDING ENERGIES\n")

print(
    f"Nb metal 3d5/2: {metal_shifted:.3f} eV, "
    f"FWHM = {metal_fwhm_reported:.3f} eV"
)
print(
    f"NbO      3d5/2: {nbo_shifted:.3f} eV, "
    f"FWHM = {shared_oxide_fwhm:.3f} eV"
)
print(
    f"NbO2     3d5/2: {nbo2_shifted:.3f} eV, "
    f"FWHM = {shared_oxide_fwhm:.3f} eV"
)
print(
    f"Nb2O5    3d5/2: {nb2o5_shifted:.3f} eV, "
    f"FWHM = {shared_oxide_fwhm:.3f} eV"
)

print(f"\nApplied energy shift = {energy_shift:+.3f} eV")

In [ ]:

# ============================================================
# RAW AREA VERIFICATION
# ============================================================

dense_components = component_curves(AREA_GRID, popt)
dense_species_total = np.sum(dense_components, axis=0)
dense_baseline = residual_baseline(AREA_GRID, popt)

raw_areas = np.array([
    abs(simpson(curve, x=AREA_GRID))
    for curve in dense_components
])

area_table = pd.DataFrame({
    "Component": component_names,
    "Fitted area parameter": fitted_areas,
    "Raw integrated area": raw_areas,
    "Difference": raw_areas - fitted_areas,
    "Unshifted 3d5/2 (eV)": unshifted_centers,
    "Shifted 3d5/2 (eV)": shifted_centers,
    "FWHM (eV)": [
        metal_fwhm_reported,
        shared_oxide_fwhm,
        shared_oxide_fwhm,
        shared_oxide_fwhm,
    ],
    "Eta": [
        np.nan,
        shared_oxide_eta,
        shared_oxide_eta,
        shared_oxide_eta,
    ],
})

display(area_table)

corrected_data_signed_area = float(simpson(y, x=x))
baseline_signed_area = float(simpson(dense_baseline, x=AREA_GRID))

print(f"\nSigned corrected-data area = {corrected_data_signed_area:.2f}")
print(f"Sum of raw species areas  = {raw_areas.sum():.2f}")
print(f"Residual baseline area    = {baseline_signed_area:.2f}")
print(
    "Species + residual baseline = "
    f"{raw_areas.sum() + baseline_signed_area:.2f}"
)


In [ ]:

# ============================================================
# SHIFTED BACKGROUND-CORRECTED FIT PLOT
# ============================================================

model_at_x = total_model(x, popt)
components_at_x = component_curves(x, popt)
baseline_at_x = residual_baseline(x, popt)

plt.figure(figsize=(11, 7))

plt.plot(
    x_shifted,
    y,
    "o",
    markersize=3,
    label="Shifted Shirley-corrected data",
)

plt.plot(
    x_shifted,
    model_at_x,
    linewidth=2.5,
    label="Total simultaneous fit",
)

plt.plot(
    x_shifted,
    baseline_at_x,
    linestyle=":",
    linewidth=1.8,
    label="Residual baseline correction",
)

for name, curve in zip(component_names, components_at_x):
    plt.plot(
        x_shifted,
        curve + baseline_at_x,
        "--",
        linewidth=1.6,
        label=name,
    )

plt.xlabel("Shifted Binding Energy (eV)")
plt.ylabel("Shirley-corrected intensity")
plt.title(f"{base_name} Advanced Simultaneous Nb 3d Fit")
plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# RAW DATA, SHIRLEY BACKGROUND, SPECIES, AND SHIFTED FIT
# ============================================================

valid_bg = np.isfinite(shirley)
bg_order = np.argsort(x_original[valid_bg])

shirley_at_x = np.interp(
    x,
    x_original[valid_bg][bg_order],
    shirley[valid_bg][bg_order],
)

raw_x_shifted = x_original + energy_shift
shirley_x_shifted = x_original + energy_shift

components_plus_background = [
    curve + baseline_at_x + shirley_at_x
    for curve in components_at_x
]

total_fit_with_background = model_at_x + shirley_at_x

plt.figure(figsize=(11, 7))

plt.plot(
    raw_x_shifted,
    y_original,
    label="Raw data",
    linewidth=1.2,
)

plt.plot(
    shirley_x_shifted,
    shirley,
    label="Shirley background",
    linewidth=2,
)

for name, curve in zip(
    component_names,
    components_plus_background,
):
    plt.plot(
        x_shifted,
        curve,
        "--",
        linewidth=1.5,
        label=name,
    )

plt.plot(
    x_shifted,
    total_fit_with_background,
    label="Total fit + Shirley background",
    linewidth=2.5,
)

plt.xlabel("Shifted Binding Energy (eV)")
plt.ylabel("Intensity")
plt.title(f"{base_name}: Raw Spectrum with Full Species Fit")
plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# RESIDUALS, UNCERTAINTIES, AND PARAMETER CORRELATIONS
# ============================================================

residuals = y - total_model(x, popt)

plt.figure(figsize=(11, 4))
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.plot(x_shifted, residuals, linewidth=1.3)
plt.xlabel("Shifted Binding Energy (eV)")
plt.ylabel("Residual")
plt.title(f"{base_name} Fit Residuals")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()

jacobian = best_result.jac
degrees_of_freedom = max(len(y) - len(popt), 1)

data_residual_variance = float(
    np.sum(data_residual_vector(popt) ** 2)
    / degrees_of_freedom
)

try:
    covariance = np.linalg.pinv(jacobian.T @ jacobian)
    covariance *= data_residual_variance

    parameter_std = np.sqrt(
        np.clip(np.diag(covariance), 0.0, None)
    )

    std_outer = np.outer(parameter_std, parameter_std)

    correlation = np.divide(
        covariance,
        std_outer,
        out=np.full_like(covariance, np.nan),
        where=std_outer > 0,
    )

except np.linalg.LinAlgError:
    covariance = np.full((len(popt), len(popt)), np.nan)
    correlation = np.full_like(covariance, np.nan)
    parameter_std = np.full_like(popt, np.nan)


metal_fwhm_std = (
    parameter_std[2] * METAL_FWHM_CALIBRATION
)

uncertainty_table = pd.DataFrame({
    "Component": component_names,
    "Area": fitted_areas,
    "Area 1-sigma": parameter_std[area_indices],
    "Shifted position (eV)": shifted_centers,
    "Position 1-sigma (eV)": parameter_std[center_indices],
    "FWHM (eV)": [
        metal_fwhm_reported,
        shared_oxide_fwhm,
        shared_oxide_fwhm,
        shared_oxide_fwhm,
    ],
    "FWHM 1-sigma (eV)": [
        metal_fwhm_std,
        parameter_std[3],
        parameter_std[3],
        parameter_std[3],
    ],
    "Eta": [
        np.nan,
        shared_oxide_eta,
        shared_oxide_eta,
        shared_oxide_eta,
    ],
    "Eta 1-sigma": [
        np.nan,
        parameter_std[4],
        parameter_std[4],
        parameter_std[4],
    ],
})

display(uncertainty_table)

parameter_names = [
    "Nb metal area",
    "Nb metal center",
    "Nb metal internal width",
    "Shared oxide FWHM",
    "Shared oxide eta",
    "NbO area",
    "NbO center",
    "NbO2 area",
    "NbO2 center",
    "Nb2O5 area",
    "Nb2O5 center",
    "Residual offset",
    "Residual slope",
]

correlation_table = pd.DataFrame(
    correlation,
    index=parameter_names,
    columns=parameter_names,
)

display(correlation_table.round(3))

nbo_nbo2_area_correlation = correlation[5, 7]

print(f"Residual standard deviation = {np.std(residuals):.6f}")
print(
    "NbO/NbO2 area correlation = "
    f"{nbo_nbo2_area_correlation:+.4f}"
)
print(
    "A value near -1 indicates strong exchange between the "
    "overlapping NbO and NbO2 areas."
)


In [ ]:
# ============================================================
# MULTI-START STABILITY DIAGNOSTIC
# ============================================================

top_n = min(10, len(candidates))
top_results = [candidates[i][3] for i in range(top_n)]

top_area_solutions = np.array([
    result.x[area_indices]
    for result in top_results
])

top_shifted_position_solutions = np.array([
    shifted_positions_from_parameters(result.x)
    for result in top_results
])

stability_table = pd.DataFrame({
    "Component": component_names,
    "Best-fit area": fitted_areas,
    f"Mean area across best {top_n} starts": np.mean(
        top_area_solutions,
        axis=0,
    ),
    f"Area std. dev. across best {top_n} starts": np.std(
        top_area_solutions,
        axis=0,
    ),
    f"Shifted-position std. dev. across best {top_n} starts": np.std(
        top_shifted_position_solutions,
        axis=0,
    ),
})

display(stability_table)

print(
    "Large area variation across similarly good starts indicates that "
    "the overlapping components remain non-unique even when the plotted "
    "total fit looks nearly unchanged."
)

In [ ]:

# ============================================================
# RAW AREA UNDER EACH FINAL FITTED COMPONENT
# ============================================================

metal_curve, nbo_curve, nbo2_curve, nb2o5_curve = component_curves(
    AREA_GRID,
    popt,
)

raw_areas = np.array([
    abs(simpson(metal_curve, x=AREA_GRID)),
    abs(simpson(nbo_curve, x=AREA_GRID)),
    abs(simpson(nbo2_curve, x=AREA_GRID)),
    abs(simpson(nb2o5_curve, x=AREA_GRID)),
])

raw_area_table = pd.DataFrame({
    "Component": component_names,
    "Raw integrated area": raw_areas,
    "Fitted area parameter": fitted_areas,
    "Difference": raw_areas - fitted_areas,
})

display(raw_area_table)

print("\nRAW AREA UNDER EACH FITTED CURVE\n")
for name, area in zip(component_names, raw_areas):
    print(f"{name:<8} = {area:.2f}")

print("--------------------------------")
print(f"Sum      = {raw_areas.sum():.2f}")


In [ ]:
# ============================================================
# ADAPTIVE SHIRLEY BACKGROUND SENSITIVITY DIAGNOSTIC
#
# This does not alter the selected fit. It checks how much the corrected
# spectrum changes under modest, automatic endpoint/window perturbations.
# ============================================================

sorted_x_original = np.sort(np.unique(x_original[np.isfinite(x_original)]))
if len(sorted_x_original) > 1:
    typical_step = float(np.median(np.diff(sorted_x_original)))
else:
    typical_step = 0.1

background_trials = []
for endpoint_points_trial in (3, 5, 7):
    for endpoint_delta_steps in (-2, 0, 2):
        trial_min = SHIRLEY_MIN_BE + endpoint_delta_steps * typical_step
        trial_max = SHIRLEY_MAX_BE - endpoint_delta_steps * typical_step

        if trial_min >= trial_max:
            continue

        try:
            trial_background = shirley_background(
                x_original,
                y_for_analysis,
                fit_min=trial_min,
                fit_max=trial_max,
                endpoint_points=endpoint_points_trial,
            )

            trial_corrected = y_for_analysis - trial_background
            trial_mask = (
                (x_original >= trial_min)
                & (x_original <= trial_max)
                & np.isfinite(trial_corrected)
            )

            if np.count_nonzero(trial_mask) < 4:
                continue

            trial_area = pchip_integral(
                x_original[trial_mask],
                trial_corrected[trial_mask],
            )

            background_trials.append({
                "Endpoint points": endpoint_points_trial,
                "Window min (eV)": trial_min,
                "Window max (eV)": trial_max,
                "Corrected total area": trial_area,
            })

        except (ValueError, RuntimeError, FloatingPointError):
            pass

background_sensitivity_table = pd.DataFrame(background_trials)
display(background_sensitivity_table)

if len(background_sensitivity_table) > 1:
    background_area_cv = (
        background_sensitivity_table["Corrected total area"].std(ddof=1)
        / background_sensitivity_table["Corrected total area"].mean()
    )
    print(
        "Relative Shirley-corrected total-area sensitivity = "
        f"{100.0 * background_area_cv:.2f}%"
    )
    print(
        "A large value means that uncertainty in the adaptive background "
        "may be comparable to the smaller NbO or NbO2 contributions."
    )
else:
    print("Not enough valid Shirley perturbations for a sensitivity estimate.")

# Optional identifiability diagnostics


In [ ]:

# ============================================================
# OPTIONAL PROFILE-LIKELIHOOD DIAGNOSTIC FOR NbO AND NbO2
#
# This diagnostic does not alter the chosen fit. It checks whether
# substantially different NbO or NbO2 areas can produce almost the
# same optimized spectrum.
# ============================================================

RUN_PROFILE_LIKELIHOOD = False

if RUN_PROFILE_LIKELIHOOD:
    profile_records = []

    for component_name, area_index in (
        ("NbO", 5),
        ("NbO2", 7),
    ):
        center_area = popt[area_index]
        tested_areas = np.linspace(
            max(0.0, 0.50 * center_area),
            1.50 * center_area,
            11,
        )

        for fixed_area in tested_areas:
            local_lower = lower.copy()
            local_upper = upper.copy()

            epsilon = max(1e-8, 1e-10 * max(abs(fixed_area), 1.0))
            local_lower[area_index] = max(0.0, fixed_area - epsilon)
            local_upper[area_index] = fixed_area + epsilon

            trial = np.clip(
                popt.copy(),
                local_lower + 1e-12,
                local_upper - 1e-12,
            )

            trial[area_index] = fixed_area

            try:
                result_profile = least_squares(
                    residual_vector,
                    trial,
                    bounds=(local_lower, local_upper),
                    method="trf",
                    x_scale="jac",
                    loss="soft_l1",
                    f_scale=1.0,
                    max_nfev=6000,
                    ftol=1e-8,
                    xtol=1e-8,
                    gtol=1e-8,
                )

                profile_records.append({
                    "Component": component_name,
                    "Fixed area": fixed_area,
                    "Relative area": fixed_area / center_area,
                    "Data score": np.mean(
                        data_residual_vector(result_profile.x) ** 2
                    ),
                })

            except (ValueError, RuntimeError, FloatingPointError):
                pass

    profile_table = pd.DataFrame(profile_records)
    display(profile_table)

    for component_name in ("NbO", "NbO2"):
        subset = profile_table[
            profile_table["Component"] == component_name
        ]

        plt.figure(figsize=(7, 4))
        plt.plot(
            subset["Relative area"],
            subset["Data score"],
            marker="o",
        )
        plt.xlabel("Fixed area / best-fit area")
        plt.ylabel("Refitted data score")
        plt.title(f"{component_name} Profile Likelihood")
        plt.tight_layout()
        plt.show()
else:
    print(
        "Profile likelihood is available but disabled. "
        "Set RUN_PROFILE_LIKELIHOOD = True to run it."
    )


In [ ]:

# ============================================================
# OPTIONAL RESIDUAL BOOTSTRAP
#
# Refits synthetic spectra generated from the fitted model plus
# resampled residuals. This estimates practical area stability.
# ============================================================

RUN_BOOTSTRAP = False
NUMBER_BOOTSTRAP_REPLICATES = 40

if RUN_BOOTSTRAP:
    bootstrap_rng = np.random.default_rng(1234)
    bootstrap_parameters = []

    fitted_signal = total_model(x, popt)
    centered_residuals = residuals - np.mean(residuals)

    for replicate in range(NUMBER_BOOTSTRAP_REPLICATES):
        synthetic_y = (
            fitted_signal
            + bootstrap_rng.choice(
                centered_residuals,
                size=len(centered_residuals),
                replace=True,
            )
        )

        def bootstrap_data_residual(p):
            return (
                region_weights
                * (total_model(x, p) - synthetic_y)
                / sigma
            )

        def bootstrap_residual(p):
            return np.concatenate([
                bootstrap_data_residual(p),
                PRIOR_STRENGTH * position_prior_residuals(p),
                PRIOR_STRENGTH * shape_prior_residuals(p),
                0.50 * baseline_prior_residuals(p),
            ])

        try:
            result_bootstrap = least_squares(
                bootstrap_residual,
                popt,
                bounds=(lower, upper),
                method="trf",
                x_scale="jac",
                loss="soft_l1",
                f_scale=1.0,
                max_nfev=5000,
                ftol=1e-8,
                xtol=1e-8,
                gtol=1e-8,
            )

            if np.all(np.isfinite(result_bootstrap.x)):
                bootstrap_parameters.append(result_bootstrap.x)

        except (ValueError, RuntimeError, FloatingPointError):
            pass

    bootstrap_parameters = np.asarray(bootstrap_parameters)

    if len(bootstrap_parameters) > 1:
        bootstrap_area_std = np.std(
            bootstrap_parameters[:, area_indices],
            axis=0,
            ddof=1,
        )

        bootstrap_table = pd.DataFrame({
            "Component": component_names,
            "Best-fit area": fitted_areas,
            "Bootstrap area 1-sigma": bootstrap_area_std,
            "Bootstrap relative 1-sigma (%)":
                100.0 * bootstrap_area_std / fitted_areas,
        })

        display(bootstrap_table)
    else:
        print("Not enough successful bootstrap fits.")
else:
    print(
        "Residual bootstrap is available but disabled. "
        "Set RUN_BOOTSTRAP = True to run it."
    )
